In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text) 

In [4]:
dataset = generate_dataset()
# dataset
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [8]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extractor from ARN\n\nHere's a comprehensive solution:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    \"\"\"\n    Extract the AWS account ID from an ARN string.\n    \n    Args:\n        arn: An AWS ARN string\n        \n    Returns:\n        The 12-digit account ID if present, None otherwise\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n        \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0')\n        '123456789012'\n        \n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket')\n        None\n        \n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:role/MyRole')\n        '123456789012'\n    \"\"\"\n    # Validate basic ARN format\n    if not arn or not arn.startswith('arn:'):\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # Split the ARN into its components\n    